# Sentiment model benchmark (week 9) and finance-aware layer (week 10)

Compares the three tone models on `documents_scored.parquet`:

| | model | coverage |
|---|---|---|
| A | VADER (lexicon) | every document |
| B | DistilBERT SST-2 (generic transformer, **binary**: no neutral class) | relevant docs |
| C | FinBERT (ProsusAI/finbert, finance transformer) | relevant docs |

then shows why tone alone is the wrong target for Treasury yields (layer 2: yield direction).

Re-run after `src/nlp/pipeline.py`. Run from the project root (`jupyter lab` there) or from `notebooks/`.

In [ ]:
import os, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

INK, MUTED, GRID, BASE, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
MODEL_COLORS = {"vader": "#2a78d6", "distilbert": "#eb6834", "finbert": "#1baf7a"}  # validated slots 1-3
MODEL_NAMES = {"vader": "A · VADER", "distilbert": "B · DistilBERT", "finbert": "C · FinBERT"}

def style(ax, title=None):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(BASE)
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, loc="left", fontsize=10, color=INK)

docs = pd.read_parquet("data/processed/documents_scored.parquet")
print(f"{len(docs):,} documents, {docs.shape[1]} columns")

## 1. Coverage

Transformers only score `is_relevant` documents (news, macro topic, or yield cue) to keep CPU time tractable.

In [ ]:
cov = pd.DataFrame({
    m: docs.groupby("source_type")[f"{m}_sentiment_score"].apply(lambda s: s.notna().mean())
    for m in MODEL_NAMES
}).rename(columns=MODEL_NAMES)
cov.insert(0, "docs", docs.groupby("source_type").size())
cov.insert(1, "relevant", docs.groupby("source_type")["is_relevant"].mean())
cov.style.format({"docs": "{:,}", **{c: "{:.1%}" for c in cov.columns[1:]}})

## 2. Score distributions

`sentiment_score = p_positive − p_negative`, on the common set of documents scored by all three models.

In [ ]:
common = docs.dropna(subset=[f"{m}_sentiment_score" for m in MODEL_NAMES])
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
fig.patch.set_facecolor(SURFACE)
bins = np.linspace(-1, 1, 41)
for ax, m in zip(axes, MODEL_NAMES):
    ax.hist(common[f"{m}_sentiment_score"], bins=bins, color=MODEL_COLORS[m], edgecolor=SURFACE, linewidth=0.6)
    style(ax, MODEL_NAMES[m])
    ax.set_xlabel("sentiment_score", color=MUTED, fontsize=9)
axes[0].set_ylabel("documents", color=MUTED, fontsize=9)
fig.suptitle(f"Sentiment score distribution, {len(common):,} docs scored by all models", x=0.01, ha="left", color=INK)
fig.tight_layout(); plt.show()

DistilBERT has no neutral class, so its probability mass is pushed to ±1 even for factual headlines; FinBERT puts most finance text near 0 (neutral), which is the realistic prior for news wires and Fed releases.

## 3. Agreement between models

In [ ]:
def label(df, m):
    p = df[[f"{m}_p_positive", f"{m}_p_neutral", f"{m}_p_negative"]].to_numpy()
    return pd.Series(np.array(["positive", "neutral", "negative"])[p.argmax(1)], index=df.index)

labels = pd.DataFrame({m: label(common, m) for m in MODEL_NAMES})
corr = common[[f"{m}_sentiment_score" for m in MODEL_NAMES]].corr(method="spearman")
corr.index = corr.columns = [MODEL_NAMES[m] for m in MODEL_NAMES]
print("Spearman correlation of sentiment_score:"); display(corr.round(3))
print("Label agreement rate (argmax class):")
display(pd.DataFrame({
    "VADER vs FinBERT": [(labels.vader == labels.finbert).mean()],
    "DistilBERT vs FinBERT": [(labels.distilbert == labels.finbert).mean()],
    "VADER vs DistilBERT": [(labels.vader == labels.distilbert).mean()],
}).round(3))
pd.crosstab(labels.finbert, labels.distilbert, rownames=["FinBERT"], colnames=["DistilBERT"])

## 4. Sample outputs

In [ ]:
cols = ["source_name", "headline"] + [f"{m}_sentiment_score" for m in MODEL_NAMES] + ["finbert_confidence", "finbert_entropy"]
common.sample(12, random_state=7)[cols].round(2)

### Largest disagreements: FinBERT vs DistilBERT

In [ ]:
gap = (common["finbert_sentiment_score"] - common["distilbert_sentiment_score"])
common.assign(gap=gap).loc[gap.abs().sort_values(ascending=False).index[:12], cols[:2] + ["finbert_sentiment_score", "distilbert_sentiment_score"]].round(2)

## 5. Layer 1 (tone) vs layer 2 (yield direction)

A headline like *“Treasuries rally on recession fears”* is **negative** in tone but implies **lower yields**. If tone were a good proxy for yield direction, the two layers would line up.

In [ ]:
rel = docs[docs["finbert_sentiment_score"].notna()]
tone = label(rel, "finbert")
pd.crosstab(tone, rel["yield_direction_label"], normalize="index").round(3).style.format("{:.1%}")

In [ ]:
ex = rel[(tone == "negative") & (rel["yield_direction_label"] == "down") & (rel["source_type"] == "news")]
ex[["headline", "finbert_sentiment_score", "yield_direction_signal", "yield_rule_categories"]].head(10).round(2)

In [ ]:
ex = rel[(tone == "negative") & (rel["yield_direction_label"] == "up") & (rel["source_type"] == "news")]
ex[["headline", "finbert_sentiment_score", "yield_direction_signal", "yield_rule_categories"]].head(10).round(2)

### Against the silver labels

Labels in `data/labels/yield_direction_labels.csv` are **LLM-assigned silver labels** (see `docs/label_taxonomy.md`). Mean tone by labeled yield direction: if tone carried the yield signal, `up` and `down` would separate cleanly.

In [ ]:
lab = pd.read_csv("data/labels/yield_direction_labels.csv")
lt = docs.merge(lab[["doc_id", "label"]], on="doc_id")
summary = lt.groupby("label")[[f"{m}_sentiment_score" for m in MODEL_NAMES] + ["yield_direction_signal"]].mean()
summary.columns = [MODEL_NAMES[m] for m in MODEL_NAMES] + ["yield_direction_signal"]
summary.insert(0, "n", lt.groupby("label").size())
summary.round(3)

`yield_direction_signal` here is in-sample (the classifier was fit on these labels); see `docs/label_taxonomy.md` for the cross-validated comparison (classifier vs. rules).

## 6. Takeaways

Filled from the numbers above when the notebook was last run:

In [ ]:
print(f"- Docs: {len(docs):,}; transformer-scored: {docs['finbert_sentiment_score'].notna().sum():,}")
print(f"- Spearman FinBERT vs DistilBERT: {corr.iloc[2,1]:.2f}; FinBERT vs VADER: {corr.iloc[2,0]:.2f}")
print(f"- FinBERT labels neutral: {(labels.finbert=='neutral').mean():.0%} of common docs; DistilBERT cannot (binary)")
print(f"- Negative-tone relevant docs labeled yield 'down': {((tone=='negative') & (rel['yield_direction_label']=='down')).sum():,}")